# Gemini top-3 history similarity + soft freshness

Compare the existing frozen Gemini top-3-history retriever with freshness-only retrieval and a small, predeclared age-penalty grid. There is no transformer fine-tuning, neural training or embedding generation in this notebook.

All variants use the same validation requests and full eligible article catalogue. Publication eligibility stays unchanged: known publication at or before each request, with no additional age cutoff. Freshness changes scores rather than removing older articles. The Stage 2 business reranker is separate.

Formula: `score = similarity - lambda * log1p(age_hours)`, where `age_hours = request_time - publication_time`. Freshness-only uses `-log1p(age_hours)`. The grid includes lambda 0, 0.01, 0.03 and 0.1. A candidate's similarity is the average of its three strongest similarities to the user's latest 10 available past-click events. History events are selected before missing vectors are removed, following the existing pipeline.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
EXPERIMENT_ID = '3.5_gemini_freshness'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    print('Upload BOTH catalogue_pipeline.py and freshness_pipeline.py from this experiment folder.')
    uploaded = files.upload()
    for module in ['catalogue_pipeline', 'freshness_pipeline']:
        matches = [name for name in uploaded if Path(name).stem.startswith(module) and Path(name).suffix == '.py']
        if len(matches) != 1: raise ValueError(f'Upload exactly one {module}.py')
        (CODE_DIR / f'{module}.py').write_bytes(uploaded[matches[0]])
else:
    choices = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in choices if (p / 'freshness_pipeline.py').is_file()), None)
    if CODE_DIR is None: raise FileNotFoundError('Open from the repository root or this experiment folder.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=1.26', 'pandas>=2.2',
                'pyarrow>=15', 'scikit-learn>=1.4'], check=True)
import pandas as pd

def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE_DIR / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
cp = load_module('catalogue_pipeline')
fp = load_module('freshness_pipeline')
print('Loaded:', cp.PIPELINE_VERSION, fp.VERSION)


Mounted at /content/drive
Upload BOTH catalogue_pipeline.py and freshness_pipeline.py from this experiment folder.


## Configuration

Use the same smaller shared catalogue and Gemini embedding inputs as your existing 0.054 Recall@200 top-3 run. The notebook reuses those saved vectors. If needed, set the exact embedding paths in CONFIG['paths']; the filename itself does not establish the encoder identity. Confirm the shared model/revision with the preprocessing owner.

Keep seed 42, latest-10 history, top-3 aggregation and the same 1,000 validation request IDs. Use a fresh run ID for changed settings. RUN_PHASE starts in audit mode. After inspecting costs, set RUN_PHASE='run' in a separate cell without rerunning configuration. Changing configuration after preparing data requires a fresh preparation/run.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + '_freshness_seed42'
RUN_PHASE = 'audit'
WORK = CODE_DIR / 'outputs' / RUN_ID
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'run_id': RUN_ID, 'owner': '',
    'drive_root': str(DRIVE_ROOT), 'seed': 42, 'scope': 'development',
    'input_version': None,
    'embedding_source': {'provider': 'Gemini', 'model': None, 'revision': None},
    'embedding_column': 'text_embedding', 'embedding_dimension': None,
    'history_limit': 10, 'history_aggregation': 'topk_history', 'history_top_k': 3,
    'validation_requests': 1000,
    'export_requests': {'train': 25, 'validation': 1000, 'test': 25},
    'freshness_weights': [0.0, 0.01, 0.03, 0.1],
    'catalogue_batch_size': 4096,
    'similarity_device': 'auto', 'similarity_min_gpu_batch': 256,
    'hash_inputs': False, 'max_index_gb': 20,
    'max_export_pairs': 150_000_000, 'max_export_gb': 10,
    'persistent_root': str(DRIVE_ROOT / 'Experiments'),
    'paths': {},
}
# Optional: point to the EXACT files used by your existing Gemini top-3 run:
# CONFIG['paths']['embeddings_train_val'] = '/content/drive/MyDrive/.../text_embeddings.parquet'
# CONFIG['paths']['embeddings_test'] = '/content/drive/MyDrive/.../text_embeddings.parquet'
print('Drive root exists:', DRIVE_ROOT.exists())
print('Local run:', WORK)


Drive root exists: True
Local run: /content/Experiments/3.5_gemini_freshness/outputs/20261105_173405_freshness_seed42


In [ ]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Catalogue articles:', audit['catalogue_articles'])
print('Catalogue articles without vectors:', audit['catalogue_articles_without_vectors'])
print('Embedding inputs:', {k: v for k, v in PREPARED['inputs'].items() if k.startswith('embeddings_')})
print('Similarity compute:', audit['compute'])
print('Validation requests:', len(PREPARED['validation_requests']))
display(pd.DataFrame(audit['planned_exports']).T)
display(pd.DataFrame(audit['full_export_costs']).T)
cp.check_cost(PREPARED)


Similarity-only compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 63654912, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
Catalogue articles: 20738
Catalogue articles without vectors: 18
Embedding inputs: {'embeddings_train_val': PosixPath('/content/drive/.shortcut-targets-by-id/12dda65WmHsymzhEkJBBpLgujTGfx1O5J/BT4222 Shared Final Folder/train_val/article embeddings/text_embeddings.parquet'), 'embeddings_test': PosixPath('/content/drive/.shortcut-targets-by-id/12dda65WmHsymzhEkJBBpLgujTGfx1O5J/BT4222 Shared Final Folder/test/article embeddings/text_embeddings.parquet')}
Similarity compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_g

,requests,eligible_pairs,empty_pools,shortlisted_articles,estimated_csv_bytes,csv_estimate_note,catalogue_articles,missing_publication_excluded,runtime_seconds,runtime_note
train,25,463808,0,5000,282500,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
validation,1000,19461013,0,200000,11300000,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
test,25,486145,0,5000,282500,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...


,requests,eligible_pairs,empty_pools,shortlisted_articles,estimated_csv_bytes,csv_estimate_note,catalogue_articles,missing_publication_excluded,runtime_seconds,runtime_note
train,232887,4319178324,0,46577400,2631623100,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
validation,29110,566218581,0,5822000,328943000,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
test,29142,566598922,0,5828400,329304600,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...


{'pairs': 20410966,
 'estimated_csv_bytes': 11865000,
 'local_free_bytes': 119816859648}

## Compare formulas and select on validation

After the audit, run the next cell to switch phase without changing RUN_ID. Similarities are calculated once per candidate batch and reused for all weights and the freshness-only control. Every formula still ranks the complete eligible pool. Selection maximises mean known-click Recall@200, then observed-click nDCG@200. Exact ties preserve the earlier option: unmodified similarity first, increasing penalties next, freshness-only last.

Known clicks excluded by eligibility remain in recall denominators and nDCG ideals. Unshown article outcomes stay unknown. Missing similarity/profile vectors retain the original zero fallback, after which the same age penalty is applied. Freshness-only can win; that is a substantive result rather than evidence that embeddings improved. Test labels do not select the weight.


In [ ]:
RUN_PHASE = 'run'  # Run after reviewing the audit.


In [ ]:
GRID = None
if RUN_PHASE == 'run':
    GRID = fp.evaluate_grid(PREPARED)
    comparison = pd.DataFrame([
        {'variant': row['variant']['name'], 'lambda': row['variant']['weight'],
         'recall@100': row['metrics']['known_click_recall@100'],
         'ndcg@100': row['metrics']['observed_click_ndcg@100'],
         'validation_requests': row['metrics']['evaluated_requests']}
        for row in GRID['results']
    ])
    display(comparison)
    print('Selected rule:', GRID['selected']['variant'])
    print('Candidate pairs scored once:', GRID['candidate_pairs_scored_once'])
    print('Grid seconds:', round(GRID['seconds'], 2))


Freshness validation: 100 requests; 1.6s
Freshness validation: 200 requests; 3.1s
Freshness validation: 300 requests; 4.7s
Freshness validation: 400 requests; 6.3s
Freshness validation: 500 requests; 7.9s
Freshness validation: 600 requests; 9.5s
Freshness validation: 700 requests; 11.1s
Freshness validation: 800 requests; 12.7s
Freshness validation: 900 requests; 14.3s
Freshness validation: 1,000 requests; 16.0s


,variant,lambda,recall@100,ndcg@100,validation_requests
0,similarity_only,0.00,0.03200,0.006319,1000
1,similarity_lambda_0p01,0.01,0.33600,0.066721,1000
2,similarity_lambda_0p03,0.03,0.82475,0.205388,1000
3,similarity_lambda_0p1,0.10,0.90275,0.255732,1000
4,freshness_only,NaN,0.90900,0.249783,1000


Selected rule: {'name': 'freshness_only', 'kind': 'freshness_only', 'weight': None}
Candidate pairs scored once: 19461013
Grid seconds: 15.97


## Export the selected rule

Only the validation-selected rule is exported, with one row per request and up to 200 IDs. `scores` are the FINAL ranking values; `similarity_scores` are the unmodified top-3 history cosines and are diagnostic. Because freshness changes order, the raw similarities need not be descending. `article_age_hours` is aligned with each article ID. Final scores can be below -1 and are not click probabilities.

Each saved shortlist is independently recomputed against the full eligible pool to verify membership, order, aligned scores and metadata. Exported-validation metrics are then calculated from the saved lists without another catalogue-scoring pass. They cover export_requests['validation'], which can differ from the selection sample. Increase export scope only after choosing a rule; use a fresh run/preparation and review budgets. Existing grid results are tied to their complete config and input signatures.

The test export ignores test outcomes. For final test metrics, join genuine held-out labels to the saved test lists using a separate evaluator after fixing settings. Stage 2 reranking and recorded-slate AUC are not implemented here.


In [ ]:
EXPORT = None
if RUN_PHASE == 'run' and GRID is not None:
    EXPORT = fp.export_selected(PREPARED, GRID)
    display(pd.json_normalize(EXPORT['exported_validation']))
    print('Export checks:', EXPORT['checks'])
    print('Shortlists:', WORK / 'rankings')
    print('Manifest:', WORK / 'run_config.json')


,known_click_recall@100,observed_click_ndcg@100,evaluated_requests,requests_scored,requests_without_known_clicks,known_clicks,known_clicks_excluded_by_eligibility,known_click_eligibility_coverage,eligible_only_recall@100,empty_eligible_pools,requests_with_ties_crossing_100,requests_without_usable_user_profile,outcomes,source,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.909,0.249783,1000,1000,0,1006,0,1.0,0.909,0,4,0,Unshown outcomes unknown; original known-click...,saved and independently verified validation sh...,0,0,0


Export checks: {'train': {'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'catalogue_pairs_rescored_for_verification': 463808, 'full_score_rows_exported': 0}, 'validation': {'requests_verified': 1000, 'shortlisted_articles_verified': 200000, 'catalogue_pairs_rescored_for_verification': 19461013, 'full_score_rows_exported': 0}, 'test': {'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'catalogue_pairs_rescored_for_verification': 486145, 'full_score_rows_exported': 0}}
Shortlists: /content/Experiments/3.5_gemini_freshness/outputs/20261105_173405_freshness_seed42/rankings
Manifest: /content/Experiments/3.5_gemini_freshness/outputs/20261105_173405_freshness_seed42/run_config.json


## Copy the completed bundle to Drive

This copies comparison results, per-request validation metrics, selected shortlists, the run manifest, helper code and package versions. Large generated SQLite indexes are omitted. Existing destinations are preserved; use a fresh copy suffix after a failed partial copy rather than silently overwriting another run. Copy completeness is checked by file sizes; this is not a saved-neural-model reload test because the rule has no learned weights.


In [ ]:
if EXPORT is not None:
    SAVED = fp.persist(PREPARED)
    print('Saved run:', SAVED)


Saved run: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.5_gemini_freshness/20261105_173405_freshness_seed42


## Report the conclusion

Compare similarity-only, freshness-only and the selected blend on identical requests. Report lambda, catalogue size, history settings, coverage and request count. An improvement belongs to the retrieval scoring rule, not a new embedding model or an established retention gain. The same validation set has already informed development; final held-out evaluation is separate. The shared evaluator must consume `scores` or preserve the exported order, rather than accidentally reordering by raw `similarity_scores`.
